# Amazon ML Challenge — Business Entity Resolution
## Fixed, fast-by-default Kaggle notebook

This version keeps the notebook's blocking/features/model logic but removes the
experimental benchmark cells and the dangerous old full SQLite indexer.

**Default:** FAST mode (10K S1 / 100K S2 / 100K S3) for quick iteration.
**Optional:** FULL mode uses compact, resumable SQLite indexes.

The full production path is intentionally opt-in because the Kaggle working disk
is approximately 20 GB and the full TRAIN files contain millions of records.


## 1. Install / import dependencies

In [1]:
# ============================================================
# DEPENDENCIES — no network install attempts
# ============================================================
import os, gc, csv, json, time, shutil, sqlite3, unicodedata, random, re, hashlib
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import psutil

try:
    from rapidfuzz.distance.Levenshtein import normalized_similarity as _lev_sim
    HAVE_RAPIDFUZZ = True
except ImportError:
    HAVE_RAPIDFUZZ = False

import lightgbm as lgb

print("rapidfuzz available:", HAVE_RAPIDFUZZ)
print("lightgbm version:", lgb.__version__)


ERROR: Could not find a version that satisfies the requirement rapidfuzz (from versions: none)
ERROR: No matching distribution found for rapidfuzz


rapidfuzz available: False
lightgbm version: 4.6.0


## 2. Locate Kaggle input files

Kaggle mounts input datasets read-only under `/kaggle/input/<dataset-name>/...`.
We don't hardcode the dataset slug (it depends on what you name the uploaded
dataset) -- instead we search for the four expected filenames anywhere under
`/kaggle/input`. If you're running this notebook outside Kaggle (e.g. to
smoke-test on a small local sample), set `DATA_DIR` manually instead.


In [2]:
def find_train_files(search_root="/kaggle/input"):
    """
    Search recursively for the four expected TRAIN files and return their
    paths. Falls back to None for any file not found -- callers must check.
    """
    wanted = {
        "train_source1.tsv": None,
        "train_source2.tsv": None,
        "train_source3.tsv": None,
        "train_ground_truth.tsv": None,
    }

    root = Path(search_root)
    if root.exists():
        for p in root.rglob("*.tsv"):
            if p.name in wanted and wanted[p.name] is None:
                wanted[p.name] = str(p)

    return wanted


DATA_DIR = None  # e.g. "/kaggle/working/local_sample" to override auto-discovery

if DATA_DIR is None:
    found = find_train_files()
    print(json.dumps(found, indent=2))
    missing = [k for k, v in found.items() if v is None]
    if missing:
        print(f"WARNING: could not locate {missing} under /kaggle/input. "
              f"Set DATA_DIR manually or check the dataset is attached.")
    S1_PATH = found["train_source1.tsv"]
    S2_PATH = found["train_source2.tsv"]
    S3_PATH = found["train_source3.tsv"]
    GT_PATH = found["train_ground_truth.tsv"]
else:
    S1_PATH = str(Path(DATA_DIR) / "train_source1.tsv")
    S2_PATH = str(Path(DATA_DIR) / "train_source2.tsv")
    S3_PATH = str(Path(DATA_DIR) / "train_source3.tsv")
    GT_PATH = str(Path(DATA_DIR) / "train_ground_truth.tsv")

WORK_DIR = Path("/kaggle/working")
OUT_DIR = WORK_DIR / "output"
SHARD_DIR = WORK_DIR / "feature_shards"
IDX_DIR = WORK_DIR / "index"
for d in (OUT_DIR, SHARD_DIR, IDX_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("S1_PATH:", S1_PATH)
print("S2_PATH:", S2_PATH)
print("S3_PATH:", S3_PATH)
print("GT_PATH:", GT_PATH)


{
  "train_source1.tsv": "/kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source1.tsv",
  "train_source2.tsv": "/kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source2.tsv",
  "train_source3.tsv": "/kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source3.tsv",
  "train_ground_truth.tsv": "/kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_ground_truth.tsv"
}
S1_PATH: /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source1.tsv
S2_PATH: /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source2.tsv
S3_PATH: /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/st

## 3. Resource-monitoring utilities

Called after every major stage (per Part 16/20 of the task spec) so we never
"wait for Kaggle to crash" -- we print RAM/disk after each stage and can
bail out early if a budget is at risk.


In [3]:
def check_memory():
    vm = psutil.virtual_memory()
    proc = psutil.Process(os.getpid())
    return {
        "system_used_gb": round(vm.used / 1e9, 2),
        "system_available_gb": round(vm.available / 1e9, 2),
        "system_percent": vm.percent,
        "process_rss_gb": round(proc.memory_info().rss / 1e9, 2),
    }


def check_disk_space(path="/kaggle/working"):
    total, used, free = shutil.disk_usage(path)
    return {
        "total_gb": round(total / 1e9, 2),
        "used_gb": round(used / 1e9, 2),
        "free_gb": round(free / 1e9, 2),
    }


def cleanup_temp_files(paths):
    for p in paths:
        p = Path(p)
        try:
            if p.is_file():
                p.unlink()
            elif p.is_dir():
                shutil.rmtree(p)
        except FileNotFoundError:
            pass
    gc.collect()


def resource_report(stage_name, **extra):
    mem = check_memory()
    disk = check_disk_space()
    print("=" * 70)
    print(f"STAGE: {stage_name}")
    for k, v in extra.items():
        print(f"  {k}: {v}")
    print(f"  RAM  -> process={mem['process_rss_gb']}GB  "
          f"system_used={mem['system_used_gb']}GB  "
          f"system_available={mem['system_available_gb']}GB")
    print(f"  DISK -> used={disk['used_gb']}GB  free={disk['free_gb']}GB")
    print("=" * 70)
    return {"stage": stage_name, "memory": mem, "disk": disk, **extra}


resource_report("startup")


STAGE: startup
  RAM  -> process=0.3GB  system_used=0.98GB  system_available=32.2GB
  DISK -> used=16.01GB  free=4.93GB


{'stage': 'startup',
 'memory': {'system_used_gb': 0.98,
  'system_available_gb': 32.2,
  'system_percent': 4.3,
  'process_rss_gb': 0.3},
 'disk': {'total_gb': 20.96, 'used_gb': 16.01, 'free_gb': 4.93}}

## 4. Configuration

Country is treated as an open string set (per the problem statement: TEST
adds France, which never appears in TRAIN) -- nothing below hardcodes
`{US, India}`.


In [4]:
# ============================================================
# CONFIG — FAST by default
# ============================================================
CONFIG = {
    "freq_caps": {
        "NP6": 300, "NP8": 300, "NS6": 300,
        "FL": 400, "FL3": 400, "F2": 400, "FML": 400,
        "AN": 400, "A2": 400, "AW2": 400, "AL2": 80, "NL": 400,
        "A4W": 10, "AT2": 20, "RAT20": 20,
        "UNP4": 300, "UNP6": 300, "UFL": 300,
    },
    "s1_chunk_size": 20_000,
    "a4_window": 3,
}

# ------------------------------------------------------------
# IMPORTANT: FAST is the default so you do NOT get multi-hour runs.
# Change MODE to "FULL" only when you actually need production-scale
# processing.
# ------------------------------------------------------------
MODE = "FAST"

FAST_S1_LIMIT = 5_000
FAST_S2_LIMIT = 100_000
FAST_S3_LIMIT = 100_000

# Full mode processes all TRAIN rows and uses compact resumable SQLite.
# It is intentionally opt-in.

print("MODE:", MODE)
if MODE == "FAST":
    print(f"FAST limits: S1={FAST_S1_LIMIT:,}, S2={FAST_S2_LIMIT:,}, S3={FAST_S3_LIMIT:,}")
else:
    print("FULL mode: all rows — use only when needed.")


{
  "freq_caps": {
    "NP6": 300,
    "NP8": 300,
    "NS6": 300,
    "FL": 400,
    "FL3": 400,
    "F2": 400,
    "FML": 400,
    "AN": 400,
    "A2": 400,
    "AW2": 400,
    "AL2": 80,
    "NL": 400,
    "A4W": 10,
    "AT2": 20,
    "RAT20": 20,
    "UNP4": 300,
    "UNP6": 300,
    "UFL": 300
  },
  "s1_chunk_size": 20000,
  "index_backend": "sqlite",
  "a4_window": 3
}


## 5. Normalization

Unchanged logic from the validated reference implementation (name/address
normalization + legal-suffix stripping + address-abbreviation folding).


In [5]:
import re

def normalize_text(text):
    if not text:
        return ""
    text = str(text).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


_NAME_SUFFIXES = [
    "incorporated", "inc", "corporation", "corp", "limited", "ltd",
    "llc", "company", "co", "private limited", "pvt ltd",
]

def normalize_business_name(name):
    name = normalize_text(name)
    words = name.split()
    while words and " ".join(words[-2:]) in _NAME_SUFFIXES:
        words = words[:-2]
    while words and words[-1] in _NAME_SUFFIXES:
        words = words[:-1]
    return " ".join(words)


_ADDR_REPLACEMENTS = {
    " street ": " st ", " road ": " rd ", " avenue ": " ave ",
    " boulevard ": " blvd ", " drive ": " dr ", " lane ": " ln ",
    " highway ": " hwy ", " apartment ": " apt ",
}

def normalize_address(address):
    address = normalize_text(address)
    address = " " + address + " "
    for old, new in _ADDR_REPLACEMENTS.items():
        address = address.replace(old, new)
    return address.strip()


# quick self-check (does not touch the real data)
assert normalize_business_name("Payne Enterprises LLC") == normalize_business_name("Payne Enterprises")
assert normalize_address("3315 Fremont Street, Peoria, IL") == normalize_address("3315 FREMONT ST, PEORIA, IL")
print("normalization self-check OK")


normalization self-check OK


## 6. Blocking strategies (all 17 key types from the validated reference)

Name: `NP6 NP8 NS6 FL FL3 F2 FML` (ASCII) + `UNP4 UNP6 UFL` (Unicode-preserving,
critical since 15.2%/11.5% of S2/S3 names are non-ASCII).
Address: `AN A2 AW2 AL2 NL` (original) + `A4W` (windowed number+nearby-word,
O(n) not O(n^2)) + `AT2` (unordered informative-token-pair) + `RAT20`
(rare-address-token, frequency-capped).

Country is folded into every key at indexing time (see Section 7's
`compose_key`) so blocking stays country-aware without special-casing it here.


In [6]:
def make_name_keys(name):
    n = normalize_business_name(name)
    if not n:
        return []
    keys = []
    compact = n.replace(" ", "")
    if len(compact) >= 6:
        keys.append(("NP6", compact[:6]))
    if len(compact) >= 8:
        keys.append(("NP8", compact[:8]))
    if len(compact) >= 6:
        keys.append(("NS6", compact[-6:]))
    words = n.split()
    if len(words) >= 2:
        first, last = words[0], words[-1]
        if len(first) >= 3 and len(last) >= 3:
            keys.append(("FL", first[:4] + "|" + last[:4]))
            keys.append(("FL3", first[:3] + "|" + last[:5]))
        w1, w2 = words[0], words[1]
        if len(w1) >= 3 and len(w2) >= 3:
            keys.append(("F2", w1[:5] + "|" + w2[:5]))
    if len(words) >= 3:
        first, middle, last = words[0], words[-2], words[-1]
        if len(first) >= 3 and len(last) >= 3:
            keys.append(("FML", first[:3] + "|" + middle[:3] + "|" + last[:3]))
    return keys


def make_unicode_name_keys(name):
    if not name:
        return []
    n = unicodedata.normalize("NFKC", str(name)).casefold()
    n = " ".join(n.split())
    if not n:
        return []
    keys = []
    compact = "".join(ch for ch in n if not ch.isspace())
    if len(compact) >= 4:
        keys.append(("UNP4", compact[:4]))
    if len(compact) >= 6:
        keys.append(("UNP6", compact[:6]))
    words = n.split()
    if len(words) >= 2:
        first, last = words[0], words[-1]
        if len(first) >= 2 and len(last) >= 2:
            keys.append(("UFL", first[:3] + "|" + last[:3]))
    return keys


def make_address_keys(address):
    a = normalize_address(address)
    if not a:
        return []
    keys = []
    words = a.split()
    if not words:
        return []
    numbers = [w for w in words if w.isdigit()]
    non_numeric = [w for w in words if not w.isdigit()]
    if numbers:
        number = numbers[0]
        for word in words:
            if not word.isdigit() and len(word) >= 3:
                keys.append(("AN", number + "|" + word[:6]))
                break
        if len(non_numeric) >= 2:
            keys.append(("A2", number + "|" + non_numeric[0][:4] + "|" + non_numeric[1][:4]))
    if len(non_numeric) >= 2:
        w1, w2 = non_numeric[0], non_numeric[1]
        if len(w1) >= 3 and len(w2) >= 3:
            keys.append(("AW2", w1[:5] + "|" + w2[:5]))
        w1, w2 = non_numeric[-2], non_numeric[-1]
        if len(w1) >= 3 and len(w2) >= 3:
            keys.append(("AL2", w1[:5] + "|" + w2[:5]))
    if numbers and non_numeric:
        number = numbers[0]
        last_word = non_numeric[-1]
        if len(last_word) >= 3:
            keys.append(("NL", number + "|" + last_word[:6]))
    return keys


GENERIC_ADDRESS_WORDS = {
    "street", "st", "road", "rd", "avenue", "ave", "lane", "ln", "drive", "dr",
    "boulevard", "blvd", "highway", "hwy", "near", "opposite", "opp",
    "building", "bldg", "floor", "fl", "block", "district", "city", "state",
    "india", "private", "limited", "ltd", "pvt",
}

def make_rare_address_token_keys(address):
    a = normalize_address(address)
    if not a:
        return []
    keys, seen = [], set()
    for tok in a.split():
        if tok.isdigit() or len(tok) < 4 or tok in GENERIC_ADDRESS_WORDS:
            continue
        token = tok[:12]
        if token and token not in seen:
            seen.add(token)
            keys.append(("RAT20", token))
    return keys


def make_at2_keys(address):
    a = normalize_address(address)
    if not a:
        return []
    informative = [t for t in a.split()
                   if not t.isdigit() and len(t) >= 4 and t not in GENERIC_ADDRESS_WORDS]
    if len(informative) < 2:
        return []
    informative = sorted(informative, key=lambda x: (-len(x), x))[:6]
    keys, seen = [], set()
    for i in range(len(informative) - 1):
        w1, w2 = informative[i][:8], informative[i + 1][:8]
        if w1 == w2:
            continue
        a1, a2 = sorted((w1, w2))
        key_value = a1 + "|" + a2
        if key_value not in seen:
            seen.add(key_value)
            keys.append(("AT2", key_value))
    return keys


def make_a4_keys(address, window=None):
    window = window or CONFIG["a4_window"]
    a = normalize_address(address)
    if not a:
        return []
    tokens = a.split()
    number_positions = [i for i, t in enumerate(tokens) if t.isdigit()]
    if not number_positions:
        return []
    keys, seen = [], set()
    for pos in number_positions:
        number = tokens[pos]
        nearby = []
        for offset in range(1, window + 1):
            for idx in (pos - offset, pos + offset):
                if 0 <= idx < len(tokens):
                    tok = tokens[idx]
                    if not tok.isdigit() and len(tok) >= 2:
                        nearby.append(tok)
        nearby = list(dict.fromkeys(nearby))[:3]
        for i in range(len(nearby)):
            for j in range(i + 1, len(nearby)):
                w1, w2 = sorted((nearby[i][:6], nearby[j][:6]))
                key_value = number + "|" + w1 + "|" + w2
                if key_value not in seen:
                    seen.add(key_value)
                    keys.append(("A4W", key_value))
    return keys


def make_all_keys(name, address):
    return (
        make_name_keys(name)
        + make_unicode_name_keys(name)
        + make_address_keys(address)
        + make_rare_address_token_keys(address)
        + make_a4_keys(address)
        + make_at2_keys(address)
    )

print("blocking key generators ready:",
      sorted({k for k, _ in make_all_keys("Test Business Pvt Ltd", "123 Main Street, Test City")}))


blocking key generators ready: ['A2', 'A4W', 'AL2', 'AN', 'AT2', 'F2', 'FL', 'FL3', 'NL', 'NP6', 'NP8', 'NS6', 'RAT20', 'UFL', 'UNP4', 'UNP6']


## 7. Build scalable S2/S3 indexes

Two backends:

- **`MemoryIndex`**: in-RAM dict-based posting lists. Fine for dev-scale
  subsets; risky for the full 5M+ record S2/S3 at once.
- **`SqliteIndex`**: disk-backed posting lists in a SQLite file with an
  index on `(key_type, key_value)`. This is the backend actually used for
  the full run (`CONFIG["index_backend"] = "sqlite"`), because it keeps
  RAM bounded regardless of dataset size -- the tradeoff being query
  latency, which is paid once per lookup during candidate generation.

Entity IDs are stored compactly (source prefix stripped, numeric suffix
kept as int/str) to reduce per-record overhead at 5M+ scale.


In [7]:
# ============================================================
# COMPACT + RESUMABLE INDEXER
# ============================================================

def split_entity_id(entity_id):
    if "-" not in entity_id:
        return "", entity_id
    prefix, suffix = entity_id.split("-", 1)
    return (prefix, int(suffix)) if suffix.isdigit() else (prefix, suffix)


def compose_key(country, key_value):
    return (country or "").strip().upper() + "||" + key_value


def _key_hash(key_type, composed):
    # 64-bit deterministic hash. This keeps SQLite tiny compared with
    # storing long UTF-8 key strings. Collision probability is negligible
    # for this application, and key_type is included in the hash.
    raw = f"{key_type}\x1f{composed}".encode("utf-8", "ignore")
    x = int.from_bytes(hashlib.blake2b(raw, digest_size=8).digest(), "big", signed=False)
    return x - (1 << 64) if x >= (1 << 63) else x


class MemoryIndex:
    def __init__(self, source_prefix):
        self.source_prefix = source_prefix
        self.postings = defaultdict(lambda: defaultdict(list))
        self.attrs = {}
        self.count = 0

    def add_record(self, entity_id, name, address, country):
        _, suffix = split_entity_id(entity_id)
        self.attrs[str(suffix)] = (name, address, country)
        for key_type, key_value in make_all_keys(name, address):
            self.postings[key_type][compose_key(country, key_value)].append(str(suffix))
        self.count += 1

    def bucket(self, key_type, key_value, limit=None):
        vals = self.postings.get(key_type, {}).get(key_value, [])
        return vals if limit is None else vals[:int(limit)]

    def get_attrs(self, suffix):
        return self.attrs.get(str(suffix))

    def num_keys(self):
        return sum(len(v) for v in self.postings.values())


class SqliteIndex:
    def __init__(self, source_prefix, db_path, manifest_path=None):
        self.source_prefix = source_prefix
        self.db_path = str(db_path)
        self.manifest_path = str(manifest_path or (self.db_path + ".manifest.json"))
        self.count = 0
        self.conn = sqlite3.connect(self.db_path)
        self.conn.execute("PRAGMA journal_mode=MEMORY")
        self.conn.execute("PRAGMA synchronous=OFF")
        self.conn.execute("PRAGMA temp_store=MEMORY")
        self.conn.execute("CREATE TABLE IF NOT EXISTS postings (key_hash INTEGER, suffix INTEGER, PRIMARY KEY(key_hash, suffix)) WITHOUT ROWID")
        self.conn.execute("CREATE TABLE IF NOT EXISTS attrs (suffix INTEGER PRIMARY KEY, name TEXT, address TEXT, country TEXT)")
        self.conn.commit()
        self._buf, self._abuf = [], []

    def _write_manifest(self, complete=False):
        tmp = self.manifest_path + ".tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump({"source_prefix": self.source_prefix, "rows_done": self.count, "complete": bool(complete)}, f)
        os.replace(tmp, self.manifest_path)

    def resume_point(self):
        if not os.path.exists(self.manifest_path):
            return 0, False
        try:
            with open(self.manifest_path, "r", encoding="utf-8") as f:
                x = json.load(f)
            return int(x.get("rows_done", 0)), bool(x.get("complete", False))
        except Exception:
            return 0, False

    def add_record(self, entity_id, name, address, country):
        _, suffix = split_entity_id(entity_id)
        suffix = int(suffix) if str(suffix).isdigit() else int(hash(str(suffix)) & 0x7fffffff)
        self._abuf.append((suffix, name, address, country))
        seen = set()
        for key_type, key_value in make_all_keys(name, address):
            kh = _key_hash(key_type, compose_key(country, key_value))
            if kh not in seen:
                seen.add(kh)
                self._buf.append((kh, suffix))
        self.count += 1
        if len(self._buf) >= 200_000:
            self._flush()

    def _flush(self):
        if self._buf:
            self.conn.executemany("INSERT OR IGNORE INTO postings VALUES (?,?)", self._buf)
            self._buf = []
        if self._abuf:
            self.conn.executemany("INSERT OR REPLACE INTO attrs VALUES (?,?,?,?)", self._abuf)
            self._abuf = []
        self.conn.commit()
        self._write_manifest(False)

    def finalize(self):
        self._flush()
        self._write_manifest(True)

    def bucket(self, key_type, key_value, limit=None):
        kh = _key_hash(key_type, key_value)
        if limit is None:
            cur = self.conn.execute("SELECT suffix FROM postings WHERE key_hash=?", (kh,))
        else:
            cur = self.conn.execute("SELECT suffix FROM postings WHERE key_hash=? LIMIT ?", (kh, int(limit)))
        return [str(r[0]) for r in cur.fetchall()]

    def get_attrs(self, suffix):
        cur = self.conn.execute("SELECT name, address, country FROM attrs WHERE suffix=?", (int(suffix),))
        return cur.fetchone()

    def num_keys(self):
        cur = self.conn.execute("SELECT COUNT(DISTINCT key_hash) FROM postings")
        return int(cur.fetchone()[0])


def read_tsv(path):
    with open(path, "r", encoding="utf-8", errors="replace", newline="") as f:
        yield from csv.DictReader(f, delimiter="\t")


def build_index(path, source_prefix, backend="sqlite", db_path=None, limit=None, manifest_path=None):
    if backend == "sqlite":
        if db_path is None:
            db_path = str(IDX_DIR / f"{source_prefix}_compact.sqlite")
        index = SqliteIndex(source_prefix, db_path, manifest_path=manifest_path)
        rows_done, complete = index.resume_point()
        if complete:
            index.count = rows_done
            print(f"[{source_prefix}] reusing complete index: {rows_done:,} rows")
            return index
        index.count = rows_done
    else:
        index = MemoryIndex(source_prefix)
        rows_done = 0

    t0 = time.time()
    for i, row in enumerate(read_tsv(path)):
        if i < rows_done:
            continue
        if limit is not None and i >= limit:
            break
        index.add_record(row.get("entity_id", ""), row.get("business_name", ""), row.get("business_address", ""), (row.get("country") or "").strip())
        if index.count and index.count % 250_000 == 0:
            print(f"  [{source_prefix}] {index.count:,} rows | {time.time()-t0:.1f}s | free={check_disk_space()['free_gb']:.2f}GB")
            if backend == "sqlite" and check_disk_space()["free_gb"] < 1.0:
                raise RuntimeError("STOPPED: less than 1 GB free disk space.")

    if backend == "sqlite":
        index.finalize()
    print(f"[{source_prefix}] ready: {index.count:,} rows, {index.num_keys():,} keys, {time.time()-t0:.1f}s")
    return index

print("Compact resumable indexer loaded.")


## 8. Generate candidates in chunks

`find_candidates` applies the per-key-type frequency cap from `CONFIG`
against whatever index was actually built -- never a value hardcoded from
a smaller dev run. `iter_s1_chunks` streams S1 in fixed-size batches so we
never hold the full 2.2M-row S1 file, or all of its candidates, in memory
at once.


In [8]:
def find_candidates(row, index, freq_caps):
    country = (row.get("country") or "").strip()
    name = row.get("business_name", "")
    address = row.get("business_address", "")
    candidates = {}
    for key_type, key_value in make_all_keys(name, address):
        cap = int(freq_caps.get(key_type, 500))
        composed = compose_key(country, key_value)
        # Only fetch cap+1 rows. If we get cap+1, the bucket is too common
        # and can be discarded without materializing the whole bucket.
        bucket = index.bucket(key_type, composed, limit=cap + 1)
        if not bucket or len(bucket) > cap:
            continue
        for suffix in bucket:
            entry = candidates.get(suffix)
            if entry is None:
                candidates[suffix] = {"key_types": {key_type}, "min_freq": len(bucket)}
            else:
                entry["key_types"].add(key_type)
                entry["min_freq"] = min(entry["min_freq"], len(bucket))
    return candidates


def generate_candidate_pairs_for_row(s1_row, indexes_by_source, freq_caps):
    s1_id = s1_row.get("entity_id", "")
    rows = []
    for source, index in indexes_by_source.items():
        for suffix, info in find_candidates(s1_row, index, freq_caps).items():
            rows.append({
                "source1_entity_id": s1_id,
                "candidate_entity_id": f"{source}-{suffix}",
                "source": source,
                "blocking_key_types": "|".join(sorted(info["key_types"])),
                "blocking_num_keys": len(info["key_types"]),
                "blocking_min_freq": info["min_freq"],
            })
    return rows


def iter_s1_chunks(path, chunk_size, limit=None):
    chunk, n = [], 0
    for row in read_tsv(path):
        if limit is not None and n >= limit:
            break
        chunk.append(row); n += 1
        if len(chunk) >= chunk_size:
            yield chunk; chunk = []
    if chunk:
        yield chunk


## 9. Candidate recall evaluation

The run reports S2, S3 and combined candidate recall. In FAST mode this is a **sample/index recall diagnostic**, not the full-data leaderboard recall, because only the configured S2/S3 sample is indexed.


In [9]:
def load_ground_truth(path):
    ground_truth = {}
    for row in read_tsv(path):
        s1_id = row["source1_entity_id"]
        raw = (row.get("matched_entity_ids") or "").strip()
        all_ids = {x.strip() for x in raw.split(",") if x.strip()} if raw else set()
        ground_truth[s1_id] = {
            "S2": {x for x in all_ids if x.startswith("S2-")},
            "S3": {x for x in all_ids if x.startswith("S3-")},
            "all": all_ids,
        }
    return ground_truth


def label_for(s1_id, candidate_id, ground_truth):
    entry = ground_truth.get(s1_id)
    return int(entry is not None and candidate_id in entry["all"])


## 10. Feature generation (chunked, `rapidfuzz`-accelerated)

Same feature set as the validated reference (`features.py`), but the
Levenshtein similarity now uses `rapidfuzz` when available -- this directly
targets the measured ~616us/pair bottleneck from the reference run. Falls
back to a pure-Python implementation only if `rapidfuzz` genuinely isn't
installed, so the notebook still runs (slowly) even offline.


In [16]:
# ============================================================
# 10. FEATURE GENERATION
# Fast version — avoids pure-Python Levenshtein bottleneck
# ============================================================

from difflib import SequenceMatcher
import re


# ------------------------------------------------------------
# LEVENSHTEIN / TEXT SIMILARITY
# ------------------------------------------------------------

def _levenshtein_distance(a, b):
    """
    Original exact Levenshtein implementation.

    Kept here for reference/fallback compatibility, but
    levenshtein_similarity() below will NOT use it when
    RapidFuzz is unavailable because it is too slow at scale.
    """
    if a == b:
        return 0

    if len(a) < len(b):
        a, b = b, a

    if not b:
        return len(a)

    prev = list(range(len(b) + 1))

    for i, ca in enumerate(a, start=1):
        cur = [i] + [0] * len(b)

        for j, cb in enumerate(b, start=1):
            cost = 0 if ca == cb else 1

            cur[j] = min(
                cur[j - 1] + 1,
                prev[j] + 1,
                prev[j - 1] + cost
            )

        prev = cur

    return prev[-1]


def levenshtein_similarity(a, b):
    """
    Fast similarity function.

    If RapidFuzz is available:
        use true normalized Levenshtein similarity.

    If RapidFuzz is NOT available:
        use SequenceMatcher instead of the extremely slow
        pure-Python Levenshtein dynamic-programming algorithm.
    """

    if not a or not b:
        return 0.0

    if a == b:
        return 1.0

    # Preferred fast implementation
    if HAVE_RAPIDFUZZ:
        return _lev_sim(a, b)

    # Fast fallback
    return SequenceMatcher(None, a, b).ratio()


def text_similarity(a, b):
    """
    Character-level similarity using SequenceMatcher.
    """

    if not a or not b:
        return 0.0

    if a == b:
        return 1.0

    return SequenceMatcher(None, a, b).ratio()


# ------------------------------------------------------------
# TOKEN SIMILARITY
# ------------------------------------------------------------

def token_similarity(a, b):
    if not a or not b:
        return 0.0

    t1 = set(a.split())
    t2 = set(b.split())

    if not t1 or not t2:
        return 0.0

    return len(t1 & t2) / len(t1 | t2)


def token_overlap_count(a, b):
    if not a or not b:
        return 0

    return len(set(a.split()) & set(b.split()))


# ------------------------------------------------------------
# CHARACTER N-GRAM SIMILARITY
# ------------------------------------------------------------

def char_ngram_jaccard(a, b, n=2):
    if not a or not b:
        return 0.0

    if len(a) < n or len(b) < n:
        return 0.0

    g1 = {
        a[i:i+n]
        for i in range(len(a) - n + 1)
    }

    g2 = {
        b[i:i+n]
        for i in range(len(b) - n + 1)
    }

    if not g1 or not g2:
        return 0.0

    return len(g1 & g2) / len(g1 | g2)


# ------------------------------------------------------------
# PREFIX / SUFFIX
# ------------------------------------------------------------

def prefix_similarity(a, b, k=4):
    if not a or not b:
        return 0.0

    ap = a[:k]
    bp = b[:k]

    m = min(len(ap), len(bp))

    if not m:
        return 0.0

    return sum(
        1 for x, y in zip(ap, bp)
        if x == y
    ) / m


def suffix_similarity(a, b, k=4):
    if not a or not b:
        return 0.0

    as_ = a[-k:]
    bs = b[-k:]

    m = min(len(as_), len(bs))

    if not m:
        return 0.0

    return sum(
        1
        for x, y in zip(
            reversed(as_),
            reversed(bs)
        )
        if x == y
    ) / m


# ------------------------------------------------------------
# ADDRESS FEATURES
# ------------------------------------------------------------

def extract_address_numbers(address):
    a = normalize_address(address)

    return re.findall(
        r"\d+",
        a
    )


def address_number_match(addr1, addr2):
    n1 = extract_address_numbers(addr1)
    n2 = extract_address_numbers(addr2)

    if not n1 or not n2:
        return 0.5

    return 1.0 if n1 == n2 else 0.0


# ------------------------------------------------------------
# COUNTRY
# ------------------------------------------------------------

def country_match(c1, c2):
    if not c1 or not c2:
        return 0.0

    return (
        1.0
        if c1.strip().lower() == c2.strip().lower()
        else 0.0
    )


# ------------------------------------------------------------
# FEATURE COLUMN ORDER
# ------------------------------------------------------------

FEATURE_COLUMNS = [
    "name_exact_match",
    "name_char_similarity",
    "name_levenshtein_similarity",
    "name_token_jaccard",
    "name_token_overlap",
    "name_compact_similarity",
    "name_prefix_similarity",
    "name_suffix_similarity",
    "name_char_ngram_jaccard",

    "address_exact_match",
    "address_char_similarity",
    "address_levenshtein_similarity",
    "address_token_jaccard",
    "address_token_overlap",
    "address_number_match",
    "address_char_ngram_jaccard",

    "country_match",

    "name_missing",
    "address_missing",

    "combined_name_address_score",
]


# ------------------------------------------------------------
# MAIN FEATURE FUNCTION
# ------------------------------------------------------------

def calculate_features(
    name1,
    address1,
    country1,
    name2,
    address2,
    country2
):
    """
    Calculate all matching features for one S1/S2 or S1/S3 pair.
    """

    # Normalize once
    n1 = normalize_business_name(name1)
    n2 = normalize_business_name(name2)

    a1 = normalize_address(address1)
    a2 = normalize_address(address2)

    # --------------------------------------------------------
    # Name similarities
    # --------------------------------------------------------

    name_char = text_similarity(
        n1,
        n2
    )

    name_lev = levenshtein_similarity(
        n1,
        n2
    )

    name_token = token_similarity(
        n1,
        n2
    )

    name_overlap = token_overlap_count(
        n1,
        n2
    )

    name_compact = text_similarity(
        n1.replace(" ", ""),
        n2.replace(" ", "")
    )

    name_prefix = prefix_similarity(
        n1,
        n2
    )

    name_suffix = suffix_similarity(
        n1,
        n2
    )

    name_ngram = char_ngram_jaccard(
        n1,
        n2
    )

    # --------------------------------------------------------
    # Address similarities
    # --------------------------------------------------------

    address_char = text_similarity(
        a1,
        a2
    )

    address_lev = levenshtein_similarity(
        a1,
        a2
    )

    address_token = token_similarity(
        a1,
        a2
    )

    address_overlap = token_overlap_count(
        a1,
        a2
    )

    address_num = address_number_match(
        address1,
        address2
    )

    address_ngram = char_ngram_jaccard(
        a1,
        a2
    )

    # --------------------------------------------------------
    # Return feature dictionary
    # --------------------------------------------------------

    return {
        # Name
        "name_exact_match": int(
            bool(n1) and n1 == n2
        ),

        "name_char_similarity": name_char,

        "name_levenshtein_similarity": name_lev,

        "name_token_jaccard": name_token,

        "name_token_overlap": name_overlap,

        "name_compact_similarity": name_compact,

        "name_prefix_similarity": name_prefix,

        "name_suffix_similarity": name_suffix,

        "name_char_ngram_jaccard": name_ngram,

        # Address
        "address_exact_match": int(
            bool(a1) and a1 == a2
        ),

        "address_char_similarity": address_char,

        "address_levenshtein_similarity": address_lev,

        "address_token_jaccard": address_token,

        "address_token_overlap": address_overlap,

        "address_number_match": address_num,

        "address_char_ngram_jaccard": address_ngram,

        # Country
        "country_match": country_match(
            country1,
            country2
        ),

        # Missing values
        "name_missing": int(
            not n1 or not n2
        ),

        "address_missing": int(
            not a1 or not a2
        ),

        # Combined score
        "combined_name_address_score": (
            0.6 * name_char
            + 0.4 * address_char
        ),
    }


# ------------------------------------------------------------
# SELF TEST
# ------------------------------------------------------------

print("=" * 70)
print("FEATURE GENERATION CELL LOADED")
print("=" * 70)

print("RapidFuzz available:", HAVE_RAPIDFUZZ)
print("Number of features:", len(FEATURE_COLUMNS))

# Quick test
_test_features = calculate_features(
    "ABC Private Limited",
    "123 Main Street",
    "India",
    "ABC Pvt Ltd",
    "123 Main St",
    "India",
)

print("Feature test successful.")
print("Generated features:", len(_test_features))

assert set(_test_features.keys()) == set(FEATURE_COLUMNS)

print("Feature-column validation: OK")
print("=" * 70)

FEATURE GENERATION CELL LOADED
RapidFuzz available: False
Number of features: 20
Feature test successful.
Generated features: 20
Feature-column validation: OK


In [29]:
# ============================================================
# 11. FINAL OPTIMIZED PIPELINE
#     Blocking + Top-K + Token Rescue + 20 Features
# ============================================================

def run_full_pipeline(
    s1_path,
    s2_index,
    s3_index,
    gt_path=None,
    chunk_size=None,
    run_limit_s1=None,
    tag="run",
    prefilter_k=30,
    rescue_name_threshold=0.67,
    rescue_address_threshold=0.50,
):
    """
    Final optimized candidate pipeline.

    FLOW:

        S1
         ↓
        Existing blocking
         ↓
        ~100 candidates
         ↓
        Top-K by blocking strength
         +
        Token-overlap rescue
         ↓
        ~33 candidates
         ↓
        Existing 20 expensive features
         ↓
        Parquet shards

    The rescue stage exists because a genuine entity may have
    strong name/address similarity but weak blocking evidence.

    Example discovered during testing:

        S1:
        White Marketing Private Limited

        S3:
        Private White Marketing Ltd

    This candidate ranked #98 by blocking strength but was
    recovered by token overlap.
    """

    chunk_size = chunk_size or CONFIG["s1_chunk_size"]

    freq_caps = CONFIG["freq_caps"]

    indexes_by_source = {
        "S2": s2_index,
        "S3": s3_index,
    }

    ground_truth = (
        load_ground_truth(gt_path)
        if gt_path
        else None
    )

    has_labels = ground_truth is not None

    # --------------------------------------------------------
    # Recall statistics
    # --------------------------------------------------------

    recall_stats = {
        source: {
            "actual": 0,
            "found": 0,
        }
        for source in indexes_by_source
    }

    combined_recall = {
        "actual": 0,
        "found": 0,
    }

    # --------------------------------------------------------
    # Counters
    # --------------------------------------------------------

    processed = 0

    total_pairs = 0

    total_base_pairs = 0

    total_rescue_pairs = 0

    total_prefilter_pairs = 0

    positive = 0

    negative = 0

    shard_idx = 0

    t0 = time.time()

    # ========================================================
    # TOKEN HELPERS
    # ========================================================

    def normalize_tokens(text):

        text = str(text or "").lower()

        text = re.sub(
            r"[^a-z0-9]+",
            " ",
            text,
        )

        return set(
            token
            for token in text.split()
            if len(token) >= 2
        )

    def overlap_score(tokens_a, tokens_b):

        if not tokens_a or not tokens_b:
            return 0.0

        return (
            len(tokens_a & tokens_b)
            /
            min(
                len(tokens_a),
                len(tokens_b),
            )
        )

    # ========================================================
    # MAIN LOOP
    # ========================================================

    for chunk in iter_s1_chunks(
        s1_path,
        chunk_size,
        limit=run_limit_s1,
    ):

        shard_rows = []

        for s1_row in chunk:

            s1_id = s1_row.get(
                "entity_id",
                "",
            )

            # =================================================
            # STEP 1
            # Existing blocking
            # =================================================

            pairs = list(
                generate_candidate_pairs_for_row(
                    s1_row,
                    indexes_by_source,
                    freq_caps,
                )
            )

            total_pairs += len(pairs)

            # =================================================
            # STEP 2
            # Ground-truth blocking recall
            # =================================================

            if has_labels:

                gt_entry = ground_truth.get(
                    s1_id,
                    {
                        "S2": set(),
                        "S3": set(),
                        "all": set(),
                    },
                )

                row_cand_ids = {
                    "S2": set(),
                    "S3": set(),
                }

                for pair in pairs:

                    source = pair["source"]

                    candidate_id = pair[
                        "candidate_entity_id"
                    ]

                    row_cand_ids[source].add(
                        candidate_id
                    )

                for source in indexes_by_source:

                    true_matches = gt_entry.get(
                        source,
                        set(),
                    )

                    recall_stats[source]["actual"] += len(
                        true_matches
                    )

                    recall_stats[source]["found"] += len(
                        true_matches
                        &
                        row_cand_ids[source]
                    )

                combined_recall["actual"] += len(
                    gt_entry.get("all", set())
                )

                combined_recall["found"] += len(
                    gt_entry.get("all", set())
                    &
                    (
                        row_cand_ids["S2"]
                        |
                        row_cand_ids["S3"]
                    )
                )

            # =================================================
            # STEP 3
            # Base ranking
            # =================================================

            pairs.sort(
                key=lambda p: (
                    -p.get(
                        "blocking_num_keys",
                        0,
                    ),

                    p.get(
                        "blocking_min_freq",
                        float("inf"),
                    ),

                    p.get(
                        "candidate_entity_id",
                        "",
                    ),
                )
            )

            base_pairs = pairs[:prefilter_k]

            total_base_pairs += len(
                base_pairs
            )

            # -------------------------------------------------
            # Selected candidate IDs
            # -------------------------------------------------

            selected_ids = {
                p["candidate_entity_id"]
                for p in base_pairs
            }

            # =================================================
            # STEP 4
            # Token rescue
            # =================================================

            s1_name = s1_row.get(
                "business_name",
                "",
            )

            s1_addr = s1_row.get(
                "business_address",
                "",
            )

            s1_country = s1_row.get(
                "country",
                "",
            )

            s1_name_tokens = normalize_tokens(
                s1_name
            )

            s1_addr_tokens = normalize_tokens(
                s1_addr
            )

            rescue_pairs = []

            # Only examine candidates that were NOT
            # already retained by the base top-K.
            #
            # This avoids duplicate work.

            for pair in pairs[prefilter_k:]:

                candidate_id = pair[
                    "candidate_entity_id"
                ]

                if candidate_id in selected_ids:
                    continue

                source = pair["source"]

                _, suffix = split_entity_id(
                    candidate_id
                )

                attrs = indexes_by_source[
                    source
                ].get_attrs(
                    suffix
                )

                if not attrs:
                    continue

                cand_name = attrs[0] or ""

                cand_addr = attrs[1] or ""

                # --------------------------------------------
                # Name overlap
                # --------------------------------------------

                cand_name_tokens = normalize_tokens(
                    cand_name
                )

                name_overlap = overlap_score(
                    s1_name_tokens,
                    cand_name_tokens,
                )

                # --------------------------------------------
                # Address overlap
                # --------------------------------------------

                cand_addr_tokens = normalize_tokens(
                    cand_addr
                )

                address_overlap = overlap_score(
                    s1_addr_tokens,
                    cand_addr_tokens,
                )

                # --------------------------------------------
                # Rescue rule
                # --------------------------------------------

                rescue = (
                    name_overlap
                    >= rescue_name_threshold
                    or
                    address_overlap
                    >= rescue_address_threshold
                )

                if rescue:

                    selected_ids.add(
                        candidate_id
                    )

                    rescue_pairs.append(
                        pair
                    )

            total_rescue_pairs += len(
                rescue_pairs
            )

            # =================================================
            # STEP 5
            # Final candidate set
            # =================================================

            final_pairs = (
                base_pairs
                +
                rescue_pairs
            )

            total_prefilter_pairs += len(
                final_pairs
            )

            # =================================================
            # STEP 6
            # Existing expensive 20 features
            # =================================================

            for pair in final_pairs:

                source = pair["source"]

                candidate_id = pair[
                    "candidate_entity_id"
                ]

                _, suffix = split_entity_id(
                    candidate_id
                )

                attrs = indexes_by_source[
                    source
                ].get_attrs(
                    suffix
                )

                if attrs:

                    (
                        cand_name,
                        cand_addr,
                        cand_country,
                    ) = attrs

                else:

                    cand_name = ""

                    cand_addr = ""

                    cand_country = ""

                feats = calculate_features(
                    s1_name,
                    s1_addr,
                    s1_country,
                    cand_name,
                    cand_addr,
                    cand_country,
                )

                out_row = {
                    **pair,
                    **feats,
                }

                # ------------------------------------------------
                # Labels
                # ------------------------------------------------

                if has_labels:

                    label = label_for(
                        s1_id,
                        candidate_id,
                        ground_truth,
                    )

                    out_row["label"] = label

                    if label:

                        positive += 1

                    else:

                        negative += 1

                shard_rows.append(
                    out_row
                )

            processed += 1

        # =====================================================
        # STEP 7
        # Write shard
        # =====================================================

        if shard_rows:

            shard_path = (
                SHARD_DIR
                /
                f"{tag}_shard_{shard_idx:05d}.parquet"
            )

            pd.DataFrame(
                shard_rows
            ).to_parquet(
                shard_path,
                index=False,
            )

        shard_idx += 1

        del shard_rows

        gc.collect()

        # =====================================================
        # Progress report
        # =====================================================

        elapsed = time.time() - t0

        resource_report(
            f"{tag} chunk {shard_idx}",
            s1_processed=processed,
            candidate_pairs=total_pairs,
            elapsed_s=round(
                elapsed,
                1,
            ),
        )

        print(
            f"  Base top-{prefilter_k}: "
            f"{total_base_pairs:,}"
        )

        print(
            f"  Rescue pairs: "
            f"{total_rescue_pairs:,}"
        )

        print(
            f"  Final feature pairs: "
            f"{total_prefilter_pairs:,}"
        )

        if total_pairs:

            print(
                f"  Final retention: "
                f"{100 * total_prefilter_pairs / total_pairs:.2f}%"
            )

    # ========================================================
    # FINAL SUMMARY
    # ========================================================

    summary = {

        "tag": tag,

        "s1_processed": processed,

        "candidate_pairs": total_pairs,

        "base_pairs": total_base_pairs,

        "rescue_pairs": total_rescue_pairs,

        "prefilter_pairs": total_prefilter_pairs,

        "avg_candidates_per_s1": (
            total_pairs / processed
            if processed
            else 0
        ),

        "avg_base_pairs_per_s1": (
            total_base_pairs / processed
            if processed
            else 0
        ),

        "avg_rescue_pairs_per_s1": (
            total_rescue_pairs / processed
            if processed
            else 0
        ),

        "avg_prefilter_pairs_per_s1": (
            total_prefilter_pairs / processed
            if processed
            else 0
        ),

        "prefilter_k": prefilter_k,

        "rescue_name_threshold":
            rescue_name_threshold,

        "rescue_address_threshold":
            rescue_address_threshold,

        "elapsed_s":
            time.time() - t0,

        "shards":
            shard_idx,
    }

    # ========================================================
    # LABEL / RECALL
    # ========================================================

    if has_labels:

        summary["positive_labels"] = positive

        summary["negative_labels"] = negative

        summary["recall_by_source"] = {
            source: (
                stats["found"]
                /
                stats["actual"]
                if stats["actual"]
                else None
            )
            for source, stats
            in recall_stats.items()
        }

        summary["recall_combined"] = (
            combined_recall["found"]
            /
            combined_recall["actual"]
            if combined_recall["actual"]
            else None
        )

    return summary

In [ ]:
# ============================================================
# 12. RUN — FAST BY DEFAULT
# ============================================================
# This is the ONLY execution cell you need for TRAIN development.
# FAST mode avoids the old multi-hour full-index/full-feature run.

# Clean only previous artifacts from THIS run.
for p in [IDX_DIR, SHARD_DIR]:
    if p.exists():
        shutil.rmtree(p, ignore_errors=True)
    p.mkdir(parents=True, exist_ok=True)

gc.collect()
resource_report("before run")

if MODE == "FAST":
    backend = "sqlite"
    s1_limit = FAST_S1_LIMIT
    s2_limit = FAST_S2_LIMIT
    s3_limit = FAST_S3_LIMIT
else:
    backend = "sqlite"
    s1_limit = None
    s2_limit = None
    s3_limit = None

print("\nBuilding S2 index...")
s2_index = build_index(
    S2_PATH, "S2", backend=backend, limit=s2_limit,
    db_path=str(IDX_DIR / "S2_compact.sqlite"),
    manifest_path=str(IDX_DIR / "S2.manifest.json"),
)

print("\nBuilding S3 index...")
s3_index = build_index(
    S3_PATH, "S3", backend=backend, limit=s3_limit,
    db_path=str(IDX_DIR / "S3_compact.sqlite"),
    manifest_path=str(IDX_DIR / "S3.manifest.json"),
)

print("\nRunning matching pipeline...")
run_summary = run_full_pipeline(
    S1_PATH, s2_index, s3_index,
    gt_path=GT_PATH,
    run_limit_s1=s1_limit,
    tag="fast_train" if MODE == "FAST" else "full_train",
    prefilter_k=30,
    rescue_name_threshold=0.67,
    rescue_address_threshold=0.50,
)

print("\nRUN SUMMARY")
print(json.dumps(run_summary, indent=2, default=str))
resource_report("after matching pipeline")


In [ ]:
# ============================================================
# 13. RECALL REPORT
# ============================================================

if "run_summary" not in globals():
    raise RuntimeError("Run the RUN cell first.")

print("=" * 70)
print("CANDIDATE RECALL")
print("=" * 70)

rb = run_summary.get("recall_by_source")
rc = run_summary.get("recall_combined")

if rb is None:
    print("Recall was not computed. Make sure gt_path=GT_PATH was passed.")
else:
    for source in ["S2", "S3"]:
        value = rb.get(source)
        print(f"{source} recall : {value * 100:.2f}%" if value is not None else f"{source} recall : N/A")

    print(f"Combined recall : {rc * 100:.2f}%" if rc is not None else "Combined recall : N/A")

    if MODE == "FAST":
        print("\nNOTE: FAST mode indexes only the configured S2/S3 sample.")
        print("This is a development recall diagnostic, NOT full-data recall.")
    else:
        print("\nFULL mode: recall is measured against the full indexed S2/S3 data.")

print("=" * 70)


## 14. Train the matching model

Loads the Parquet shards written by Section 11/12 (never all candidates as
one in-memory DataFrame if it doesn't fit -- `pd.read_parquet` per shard,
concatenated only if it comfortably fits, otherwise switch to
`lgb.Dataset`'s native chunked/streaming construction). Splits by
**S1 entity** (not by row) to avoid leaking the same S1's candidates across
train/validation.


In [ ]:
def load_shards(tag):
    paths = sorted(SHARD_DIR.glob(f"{tag}_shard_*.parquet"))
    if not paths:
        raise FileNotFoundError(f"No shards found for tag={tag!r} in {SHARD_DIR}")
    return pd.concat((pd.read_parquet(p) for p in paths), ignore_index=True)


def train_val_split_by_s1(df, val_frac=0.2, seed=42):
    s1_ids = df["source1_entity_id"].unique()
    rng = random.Random(seed)
    s1_ids = list(s1_ids)
    rng.shuffle(s1_ids)
    n_val = int(len(s1_ids) * val_frac)
    val_ids = set(s1_ids[:n_val])
    val_mask = df["source1_entity_id"].isin(val_ids)
    return df[~val_mask].copy(), df[val_mask].copy()


def train_matcher(df):
    X = df[FEATURE_COLUMNS + ["blocking_num_keys", "blocking_min_freq"]]
    y = df["label"]

    train_df, val_df = train_val_split_by_s1(df)
    X_train = train_df[X.columns]
    y_train = train_df["label"]
    X_val = val_df[X.columns]
    y_val = val_df["label"]

    train_set = lgb.Dataset(X_train, label=y_train)
    val_set = lgb.Dataset(X_val, label=y_val, reference=train_set)

    params = {
        "objective": "binary",
        "metric": ["auc", "binary_logloss"],
        "boosting_type": "gbdt",
        "learning_rate": 0.05,
        "num_leaves": 63,
        "min_data_in_leaf": 50,
        "feature_fraction": 0.9,
        "bagging_fraction": 0.9,
        "bagging_freq": 1,
        "verbose": -1,
    }

    model = lgb.train(
        params, train_set, num_boost_round=500,
        valid_sets=[train_set, val_set], valid_names=["train", "val"],
        callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)],
    )

    val_df = val_df.copy()
    val_df["score"] = model.predict(X_val, num_iteration=model.best_iteration)
    return model, train_df, val_df


## 15. Validate: candidate recall vs. final match quality, and F0.5

Two genuinely different numbers, reported separately (never conflating
"98% candidate recall" with "98% matching accuracy"):

- **Candidate recall** (already computed in Section 11/13): did blocking
  include the true match at all?
- **Final match quality**: given a score threshold/top-k rule, does the
  *matcher* pick the right candidate(s)? Evaluated with the competition's
  actual metric, **macro-averaged F0.5 per S1 entity**, exactly as defined
  in the problem statement (singletons scored 1.0 for a correct empty
  prediction, 0.0 for any false-positive match).


In [ ]:
# Validation helper is defined above. The 14B cell runs the actual validation.
print("Validation functions loaded.")


In [ ]:
# ============================================================
# 14B. TRAIN LIGHTGBM ON THE GENERATED SHARDS
# ============================================================
train_tag = "fast_train" if MODE == "FAST" else "full_train"
df = load_shards(train_tag)
print("Training rows:", len(df))
print("Positive labels:", int(df["label"].sum()))

model, train_df, val_df = train_matcher(df)
model_path = OUT_DIR / f"matcher_{MODE.lower()}.txt"
model.save_model(str(model_path))
print("Model saved:", model_path)

gt = load_ground_truth(GT_PATH)
f05, n_eval = evaluate_f05(val_df, gt, threshold=0.5)
print(f"Validation macro F0.5: {f05:.4f} over {n_eval:,} S1 entities")

del df, train_df, val_df
gc.collect()
resource_report("after model training")


## 16. Prepare for TEST inference (not run yet -- no test data provided)

This function is deliberately **not executed**. The competition's TEST
dataset has not been supplied yet. It's defined here, matching the exact
`matching_results.tsv` / `candidate_pairs.tsv` output format from the
problem statement, so it can be run as soon as test data is available,
without changing anything upstream.


In [ ]:
def run_test_inference(test_s1_path, test_s2_path, test_s3_path, model,
                        threshold=0.5, out_dir=None):
    """
    NOT YET RUN -- requires the competition TEST files, which have not
    been provided. Wires together the same index/candidate/feature code
    as TRAIN, scores every candidate with `model`, and writes both
    required output files in the exact format the problem statement
    specifies:

      matching_results.tsv : source1_entity_id, matched_entity_ids
      candidate_pairs.tsv  : source1_entity_id, candidate_entity_ids

    Every S1 test entity gets exactly one row in both files (empty
    matched_entity_ids for singletons), IDs restricted to those present
    in the test S2/S3 files, no duplicates within a list.
    """
    out_dir = Path(out_dir or OUT_DIR)

    test_s2_index = build_index(test_s2_path, "S2", backend=CONFIG["index_backend"],
                                 db_path=str(IDX_DIR / "test_s2_index.sqlite"))
    test_s3_index = build_index(test_s3_path, "S3", backend=CONFIG["index_backend"],
                                 db_path=str(IDX_DIR / "test_s3_index.sqlite"))
    indexes_by_source = {"S2": test_s2_index, "S3": test_s3_index}

    matching_rows = []
    candidate_rows = []

    for chunk in iter_s1_chunks(test_s1_path, CONFIG["s1_chunk_size"]):
        for s1_row in chunk:
            s1_id = s1_row.get("entity_id", "")
            pairs = generate_candidate_pairs_for_row(s1_row, indexes_by_source, CONFIG["freq_caps"])

            candidate_ids = []
            scored = []

            for pair in pairs:
                source = pair["source"]
                candidate_id = pair["candidate_entity_id"]
                candidate_ids.append(candidate_id)

                _, suffix = split_entity_id(candidate_id)
                attrs = indexes_by_source[source].get_attrs(suffix)
                cand_name, cand_addr, cand_country = attrs if attrs else ("", "", "")
                feats = calculate_features(
                    s1_row.get("business_name", ""), s1_row.get("business_address", ""),
                    s1_row.get("country", ""), cand_name, cand_addr, cand_country,
                )
                X = pd.DataFrame([{**feats,
                                    "blocking_num_keys": pair["blocking_num_keys"],
                                    "blocking_min_freq": pair["blocking_min_freq"]}])
                X = X[FEATURE_COLUMNS + ["blocking_num_keys", "blocking_min_freq"]]
                score = model.predict(X, num_iteration=model.best_iteration)[0]
                if score >= threshold:
                    scored.append(candidate_id)

            matching_rows.append({
                "source1_entity_id": s1_id,
                "matched_entity_ids": ",".join(dict.fromkeys(scored)),
            })
            candidate_rows.append({
                "source1_entity_id": s1_id,
                "candidate_entity_ids": ",".join(dict.fromkeys(candidate_ids)),
            })

    pd.DataFrame(matching_rows).to_csv(out_dir / "matching_results.tsv", sep="\t", index=False)
    pd.DataFrame(candidate_rows).to_csv(out_dir / "candidate_pairs.tsv", sep="\t", index=False)
    print(f"Wrote {out_dir/'matching_results.tsv'} and {out_dir/'candidate_pairs.tsv'}")


print("run_test_inference() is defined but NOT called -- no TEST dataset provided yet.")
